# Brightspeed Billing & Payments — Full-Year Reprocessing (Steps 1–3)

Reprocesses the full scheduled year (Jan 2026 → Aug 24 2026) of Brightspeed call-center data, keeping **only billing & payments** lines, into aligned parquet datasets for downstream tagging and BI.

## 1. Raw string parquet (all columns aligned)

`pipeline/ingest.py` streams each monthly `*_All_Call_Data.xlsx` in read-only mode, aligns every month onto one union schema, and coerces all columns to `str`.

In [ ]:
import pandas as pd
raw = pd.read_parquet('code/data/raw_all_string.parquet')
print('raw rows/cols:', raw.shape)
print('all-string:', all(str(t) in ('object','string') for t in raw.dtypes))
raw.head(2)

## 2. Billing & Payments dataset

Keep rows whose `Primary Call Reason` is one of the billing categories.

In [ ]:
from code_local.pipeline import billing_narrow as BN
billing = pd.read_parquet('code/data/billing_all.parquet')
print('billing rows:', len(billing))
print(billing['Primary Call Reason'].value_counts().to_string())

## 3. Narrow billing/payments parquet

The exact columns used for ChatGPT-Excel tagging and the BI report.

In [ ]:
narrow = pd.read_parquet('code/data/billing_narrow.parquet')
print('narrow shape:', narrow.shape)
print(narrow.columns.tolist())
print(narrow['Call_Month'].value_counts().sort_index().to_string())

## Summary

`raw_all_string.parquet` → `billing_all.parquet` → `billing_narrow.parquet`.
The narrow dataset feeds the monthly tagging workbook (step 6) and the report (step 7).